# P1 ML Core — Voice Cloning Detection (Colab)
This notebook is your **P1 (ML Core)** workspace. Run top to bottom.

**What you own:** audio preprocessing, deepfake-audio model inference, voiceprint (speaker embedding), prosody features, and the accuracy numbers (English + Hindi).

**Flow:** install deps → test detection → enroll a voice → test voiceprint → extract prosody → measure accuracy on real vs cloned clips.


In [ ]:
# @title 1. Mount Drive + get the src/ package (results/checkpoints/test_data live on DRIVE)
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"   # change if you forked
REPO_DIR = Path("/content/VoxDetect")                    # session-only git clone of the repo

# ---- Durable on Google Drive (persist across sessions) ----
# mirror folium: RESULTS_DIR, CHECKPOINT_DIR, data all under /content/drive/MyDrive/VoxDetect
ML_BASE  = Path("/content/drive/MyDrive/VoxDetect/ml-core")
RESULTS_DIR    = ML_BASE / "results"        # every evaluate run writes a UNIQUE json here
CHECKPOINT_DIR = ML_BASE / "checkpoints"     # fine-tuned/frozen model artifacts
TEST_DATA      = ML_BASE / "test_data"       # real/ + cloned/ clips, English + Hindi
results_dir = RESULTS_DIR; test_data_dir = TEST_DATA
for d in (RESULTS_DIR, CHECKPOINT_DIR, TEST_DATA):
    d.mkdir(parents=True, exist_ok=True)

# ---- git clone the repo (session-only) so we get the latest src/ ----
if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))
print("src/ package at:", SRC_PKG)
print("RESULTS_DIR (Drive):", RESULTS_DIR)
print("CHECKPOINT_DIR (Drive):", CHECKPOINT_DIR)
print("TEST_DATA (Drive):", TEST_DATA)

# Install audio + ML deps (no stray 'audio' package)
!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title 2. Load audio + preprocess (sanity check)
from audio_utils import load_audio, preprocess_clip, chunk_audio
import numpy as np

# TEST_DATA lives on DRIVE (cell 1): test_data_dir = .../MyDrive/VoxDetect/ml-core/test_data
# Drop a clip at test_data_dir/"sample.wav" (or real/english/...) then run:
try:
    wav, sr = load_audio(str(test_data_dir / "sample.wav"))
    print("loaded", wav.shape, "at", sr, "Hz")
except Exception as e:
    print("No clip yet. Add test_data/sample.wav on Drive, then re-run:", e)

In [ ]:
# @title 3. Deepfake detection with pretrained model
# Chooses the model. For a real synthetic/real detector, swap the repo ID in detect.py
# to a deepfake-audio checkpoint (e.g. ForASD or a HF 'audio-deepfake' model).
from detect import DetectionEngine, quick_check

# This may ask for an HF token if the model is gated. Set HF_TOKEN in secrets.
eng = DetectionEngine(model_variant="wav2vec2")

# Test on a real clip -> expect LOW risk
try:
    r = eng.analyze_audio("test_data/sample.wav")
    print("sample.wav:", r["risk_score"], r["band"], r["signals"])
except Exception as e:
    print("Detection test failed (expected until a real clip + model are wired):", e)


In [ ]:
# @title 6. FIRST-RUN VALIDATION — do this before trusting any numbers
import validate

# Option A: two clips on Drive under test_data_dir (real/ and cloned/)
# validate.run(real=str(test_data_dir/"real/english/a.wav"),
#              cloned=str(test_data_dir/"cloned/english/a.wav"), threshold=70)
# Option B: just load the model and print the label mapping first
from detect import DetectionEngine
eng = DetectionEngine(model_variant="wav2vec2")
print("id2label:", eng.id2label)
print("CONFIRM which index = 'fake' before trusting any risk score.")

In [ ]:
# @title 4. Voiceprint: enroll + compare
from voiceprint import Voiceprint

vp = Voiceprint()
try:
    enrolled = vp.enroll(str(test_data_dir / "sample.wav"), label="CXO")
    print("enrolled embedding dim:", len(enrolled["embedding"]))
    emb2 = vp.embed(str(test_data_dir / "sample2.wav"))
    sim = vp.similarity(enrolled["embedding"], emb2)
    print("same-speaker similarity (should be >0.4):", round(sim, 3))
except Exception as e:
    print("Voiceprint needs a real clip on Drive. Error:", e)

In [ ]:
# @title 5. Prosody features
from prosody import extract_prosody, prosody_anomaly_score

try:
    feat = extract_prosody(str(test_data_dir / "sample.wav"))
    print("prosody:", feat)
    print("anomaly:", round(prosody_anomaly_score(feat), 3))
except Exception as e:
    print("Prosody needs a real clip on Drive. Error:", e)

In [ ]:
# @title 6. (Coming) Build your own real vs cloned dataset
# P4 will supply on Drive: test_data/{real,cloned}/{english,hindi}/...
# Then run evaluate.py for each split -> results/<sprint>.json (also on Drive).
print("Dataset build + measurement: run sprint1/sprint2 notebooks (they write to",
      "RESULTS_DIR on Drive).")